# Day 2 · 03. LangGraph 기초: 상태·노드·엣지로 워크플로 짜기

## 학습 목표
- LangGraph의 핵심 3요소(**State**, **Node**, **Edge**)가 각각 무엇이고 왜 필요한지 설명할 수 있다.
- `TypedDict`로 상태 스키마를 정의하고, 단일 노드 그래프를 `compile` → `invoke` / `stream` 할 수 있다.
- `add_conditional_edges`로 **분기 로직**을 구현하고, 왜 조건 라우팅이 에이전트의 핵심인지 말할 수 있다.
- LLM에 도구(`@tool`)를 `bind_tools`로 연결하고 `ToolNode`로 **에이전트 실행 루프**를 만들 수 있다.

## 핵심 키워드
StateGraph · TypedDict · add_messages · add_edge · add_conditional_edges · bind_tools · ToolNode · stream

## 이 노트북의 위치
01·02에서는 **무엇을** 검색할지(Hybrid, Rerank)를 다뤘다. 여기서는 검색·판단·생성 단계를 **어떤 흐름으로 엮을지** 결정하는 도구인 LangGraph를 배운다. 이 노트북의 문법이 뒤이어 나오는 04(Naive RAG) · 05(Self-RAG) · 06(Adaptive RAG) 노트북의 **공통 뼈대**가 된다.

```
앞 (01·02)                 여기 (03)                  뒤 (04·05·06)
검색 품질 끌어올리기  ─▶  워크플로 뼈대 익히기  ─▶  자가교정·라우팅 RAG로 조립
```


In [ ]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '..')

from common import get_chat_model, get_embeddings, provider_badge
from common.loaders import load_any

print(provider_badge())


## 1. 왜 LangGraph인가: 체인으로는 모자란 순간

LangChain의 LCEL 체인(`prompt | llm | parser`)은 **한 방향 직선**이다. 입력이 들어가면 파이프를 따라 한 번 흘러 결과가 나온다. 단순 Q&A에는 충분하지만, 실무 에이전트는 그렇게 단순하지 않다.

실제로 필요한 동작들:

| 필요한 동작 | 체인으로 가능? | LangGraph 해결 방식 |
|---|---|---|
| **분기** — 질문 유형에 따라 다른 경로 | ❌ 한 방향뿐 | 조건부 엣지 (`add_conditional_edges`) |
| **반복/재시도** — 결과가 부실하면 다시 검색 | ❌ 사이클 불가 | 노드 간 루프 엣지 |
| **공유 상태** — 여러 단계가 같은 데이터 참조/수정 | △ 번거로움 | 명시적 `State` 객체 |
| **도구 호출 루프** — LLM이 도구를 여러 번 써야 답 가능 | ❌ 수동 구현 | `ToolNode` + 조건부 엣지 |

### LangGraph의 세 조각

LangGraph는 이 모든 것을 세 개의 단어로 정리한다.

- **State**: 노드들 사이를 흐르는 **데이터 계약**. "이 워크플로에서 오가는 값들의 스키마".
- **Node**: State를 받아 새 State(또는 일부)를 돌려주는 **함수**. 하는 일 하나당 하나.
- **Edge**: 노드에서 노드로 가는 **화살표**. 고정 엣지 + 조건부 엣지.

> 📌 **한 줄 요약**: **"State를 공유하는 함수들(Node)이 엣지(Edge)로 이어진 워크플로"** — 이게 LangGraph다. 이 개념만 잡으면 뒤에 나오는 복잡한 예시도 전부 같은 모양이다.


## 2. State: 노드 사이를 흐르는 '데이터 계약'

State는 그냥 Python `dict` 라고 생각해도 된다. 다만 **어떤 키가 어떤 타입으로 들어올지 미리 명시**해두면 오타·타입 실수를 컴파일 타임에 잡을 수 있어 `TypedDict`를 쓴다.

### State에 무엇을 넣어야 하나?

기준은 단순하다 — **여러 노드가 공유해야 하는 값만** 넣는다. 한 노드 안에서만 쓰는 임시 변수는 넣지 말자(State가 쓸데없이 커진다).

예를 들어 RAG에서는 보통:
- `question`: 사용자가 던진 질문 (모든 노드가 읽는다)
- `documents`: 검색된 문서 (retrieve 노드가 쓰고, grade/generate가 읽는다)
- `generation`: 최종 답변 (generate 노드가 쓴다)

전부 "여러 노드가 쓰거나 읽는" 값이다.


In [ ]:
from typing import TypedDict

# 가장 단순한 State: 카운터 하나와 로그 문자열 하나
class CounterState(TypedDict):
    count: int
    message: str

# TypedDict는 런타임엔 그냥 dict다 — 아래도 정상 동작한다
sample: CounterState = {'count': 0, 'message': '시작'}
print(sample, type(sample))


### Reducer 살짝 엿보기: 여러 노드가 같은 필드를 '누적' 하고 싶다면?

기본 동작은 **덮어쓰기** — 한 노드가 `{'count': 5}`를 돌려주면 State의 `count`는 5로 바뀐다.

그런데 챗봇처럼 **"기존 메시지 뒤에 새 메시지를 붙여야 하는"** 필드는 덮어쓰기가 아니라 **누적**이 필요하다. 이때 쓰는 게 `Annotated[타입, reducer함수]` 문법이다. 5장에서 `add_messages` reducer로 직접 써본다.


## 3. Node: 상태를 받아 상태를 돌려주는 함수

노드는 시그니처가 딱 정해져 있다.

```python
def my_node(state: MyState) -> dict:
    # state를 읽어서
    # 새로운 값을 계산하고
    # 바꾸고 싶은 필드만 dict로 반환
    return {'count': state['count'] + 1}
```

전체 State를 통째로 반환하지 않아도 된다 — **바뀐 필드만 반환**하면 LangGraph가 알아서 합쳐준다. 반환 안 한 필드는 그대로 유지된다.

### 좋은 노드의 조건
- **부수효과 최소화**: 파일 쓰기, 전역 변수 수정 같은 건 피하자. 디버깅이 어렵다.
- **한 노드 한 역할**: "검색 + 평가 + 생성"을 한 노드에 몰아넣지 말고 세 노드로 쪼갠다. 나중에 조건부 엣지를 끼워넣기 쉽다.


In [ ]:
# LLM 없이 순수 파이썬 노드부터 — 그래프 역학에만 집중하자

def increment(state: CounterState) -> dict:
    '''count를 1 증가시킨다.'''
    new_count = state['count'] + 1
    return {'count': new_count}

def stamp_message(state: CounterState) -> dict:
    '''현재 count 값을 문자열에 찍어 넣는다.'''
    return {'message': f"카운트는 {state['count']} 입니다"}

# 노드는 그냥 함수다 — 직접 호출해서 동작 확인 가능
s = {'count': 0, 'message': ''}
s.update(increment(s))     # {'count': 1}
s.update(stamp_message(s)) # {'message': '카운트는 1 입니다'}
print(s)


## 4. Edge와 StateGraph: 노드를 이어붙여 실행하기

이제 노드들을 **어떤 순서로** 부를지 지정해야 한다. 그걸 담는 그릇이 `StateGraph`다.

### 주요 API 5가지

| 호출 | 의미 |
|---|---|
| `g = StateGraph(State)` | 상태 스키마로 빈 그래프 생성 |
| `g.add_node('name', func)` | 노드 등록 (이름 ↔ 함수) |
| `g.add_edge(from, to)` | 고정 엣지 — `from` 끝나면 무조건 `to`로 |
| `g.compile()` | 실행 가능한 앱(runnable)으로 컴파일 |
| `app.invoke(init)` / `app.stream(init)` | 초기 State로 시작해서 끝까지 실행 |

`START`는 "입구", `END`는 "출구"라는 이름이 붙은 특별 노드다.


In [ ]:
from langgraph.graph import StateGraph, START, END

# 1) 그래프 생성
g = StateGraph(CounterState)

# 2) 노드 등록
g.add_node('inc', increment)
g.add_node('stamp', stamp_message)

# 3) 엣지 연결: START -> inc -> stamp -> END
g.add_edge(START, 'inc')
g.add_edge('inc', 'stamp')
g.add_edge('stamp', END)

# 4) 컴파일
app = g.compile()

# 5) 초기 상태로 실행 — invoke는 최종 상태 하나만 돌려준다
final = app.invoke({'count': 0, 'message': ''})
print('invoke 결과:', final)


In [ ]:
# stream은 각 노드가 끝날 때마다 이벤트를 yield한다 — 디버깅/관찰에 유용
print('=== stream 이벤트 ===')
for event in app.stream({'count': 0, 'message': ''}):
    print(event)


> 💡 **여기까지 LLM은 한 번도 안 불렀다.** 그래도 State → Node → Edge → compile → invoke 의 실행 사이클이 완성됐다. 이게 LangGraph의 전부다 — 나머지는 노드 안에서 LLM을 부르거나, 엣지를 조건부로 만드는 것뿐이다.


## 5. 첫 LLM 그래프: 단일 노드 챗봇

챗봇의 State는 대화 히스토리, 즉 **메시지의 리스트**가 중심이다.

여기서 문제 하나: 챗봇 노드는 "새 응답 메시지 하나"를 돌려주고 싶다. 그런데 State의 `messages`는 **기존 히스토리 + 새 응답**이 되어야 한다. 덮어쓰기면 히스토리가 날아가 버린다.

### `add_messages` reducer
해결책은 LangGraph가 제공하는 `add_messages`라는 **reducer 함수**다. 필드 타입을 `Annotated[list, add_messages]` 로 선언하면, 노드가 `{'messages': [새 메시지]}` 를 반환할 때 LangGraph가 **기존 리스트에 이어붙여 준다**. 덮어쓰기가 아니라 append가 되는 것.


In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]  # 핵심: 덮어쓰기 X, 이어붙이기 O

llm = get_chat_model(temperature=0.0)

def chatbot(state: ChatState) -> dict:
    '''현재까지의 메시지를 LLM에 넣고, 응답 한 개를 반환.'''
    reply = llm.invoke(state['messages'])
    return {'messages': [reply]}

g = StateGraph(ChatState)
g.add_node('chatbot', chatbot)
g.add_edge(START, 'chatbot')
g.add_edge('chatbot', END)
chat_app = g.compile()
print('챗봇 그래프 컴파일 완료')


In [ ]:
# invoke: 최종 State 하나만 받는다
result = chat_app.invoke({
    'messages': [HumanMessage(content='전자금융거래법이 무엇인지 한 문장으로 설명해줘.')]
})
print('메시지 수:', len(result['messages']))
print('마지막 응답:', result['messages'][-1].content)


In [ ]:
# stream: 각 노드가 업데이트를 내놓을 때마다 이벤트를 받아볼 수 있다
print('=== stream 이벤트 (노드별 업데이트) ===')
for event in chat_app.stream({
    'messages': [HumanMessage(content='망분리란 무엇인가? 한 문장으로.')]
}):
    for node_name, update in event.items():
        last_msg = update['messages'][-1]
        print(f'[{node_name}] {last_msg.content[:80]}...')


**invoke vs stream 정리**
- `invoke`: 끝까지 실행하고 최종 State 한 번 반환. 단순 호출에 적합.
- `stream`: 노드 하나 끝날 때마다 이벤트를 yield. **로그·디버깅·UI 실시간 표시**에 필수.


## 6. 조건부 엣지: 흐름을 둘로 가르기

지금까지의 엣지는 모두 고정이었다(`A → B → C`). 실제 에이전트는 상태에 따라 경로를 바꾼다.

### 문법

```python
g.add_conditional_edges(
    'classify',          # 출발 노드
    route_fn,            # State를 받아 '라벨 문자열'을 반환하는 함수
    {                    # 라벨 → 실제 갈 노드 이름 매핑
        'short': 'answer_short',
        'long':  'answer_long',
    },
)
```

`route_fn(state) -> str`은 **아무 계산이나** 해도 된다 — 문자열 길이, LLM 분류 결과, 딕셔너리 키 존재 여부 등. 반환한 라벨을 기준으로 LangGraph가 다음 노드를 선택한다.

### 예시: 질문 길이로 분기

아래 그래프는 질문이 **20자 미만**이면 "간단 답변" 노드로, 이상이면 "상세 답변" 노드로 보낸다. LLM 호출은 분기 끝에서 한 번뿐이다.

```
START → classify ──(short)──▶ answer_short ─▶ END
                 └─(long)───▶ answer_long  ─▶ END
```


In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

class RouteState(TypedDict):
    messages: Annotated[list, add_messages]
    route: str   # 라우팅 결과 기록용(옵션)

# --- 노드들 ---
def classify(state: RouteState) -> dict:
    '''분류는 State만 확인 — LLM 없이 길이 기준으로 라벨만 찍는다.'''
    q = state['messages'][-1].content
    label = 'short' if len(q) < 20 else 'long'
    return {'route': label}

prompt_short = ChatPromptTemplate.from_template(
    '아래 질문에 한 문장으로만 간단히 답하라.\n질문: {q}'
)
prompt_long = ChatPromptTemplate.from_template(
    '아래 질문에 배경·정의·예시를 포함해 3~4문장으로 자세히 답하라.\n질문: {q}'
)
chain_short = prompt_short | llm
chain_long  = prompt_long  | llm

def answer_short(state: RouteState) -> dict:
    q = state['messages'][-1].content
    reply = chain_short.invoke({'q': q})
    return {'messages': [reply]}

def answer_long(state: RouteState) -> dict:
    q = state['messages'][-1].content
    reply = chain_long.invoke({'q': q})
    return {'messages': [reply]}

# --- 라우터 함수 ---
def route_fn(state: RouteState) -> str:
    return state['route']   # 'short' 또는 'long' 을 반환

# --- 그래프 조립 ---
g = StateGraph(RouteState)
g.add_node('classify', classify)
g.add_node('answer_short', answer_short)
g.add_node('answer_long',  answer_long)

g.add_edge(START, 'classify')
g.add_conditional_edges('classify', route_fn, {
    'short': 'answer_short',
    'long':  'answer_long',
})
g.add_edge('answer_short', END)
g.add_edge('answer_long',  END)

route_app = g.compile()
print('조건부 그래프 컴파일 완료')


In [ ]:
# 짧은 질문 → answer_short 로 라우팅
r1 = route_app.invoke({
    'messages': [HumanMessage(content='망분리란?')],
    'route': '',
})
print(f"[route={r1['route']}] {r1['messages'][-1].content[:120]}")

# 긴 질문 → answer_long 로 라우팅
r2 = route_app.invoke({
    'messages': [HumanMessage(content='전자금융거래법에서 정의하는 전자금융거래의 범위와 주요 규제 포인트를 설명해줘.')],
    'route': '',
})
print(f"\n[route={r2['route']}] {r2['messages'][-1].content[:200]}...")


> 💡 **조건부 엣지는 에이전트의 심장이다.** 05 Self-RAG에서는 "문서 품질이 좋은가?" 를 라우터로 써서 재검색 루프를 만들고, 06 Adaptive RAG에서는 "질문 복잡도" 로 단일 검색 vs 다단계 추론을 가른다. 원리는 전부 이 섹션과 같다.

## 7. 도구를 쓰는 에이전트: bind_tools + ToolNode

이제 "에이전트"라 부를 만한 것을 만든다. 에이전트의 핵심은 **LLM이 스스로 '도구를 쓸지 말지' 결정**한다는 점이다.

### 에이전트 루프

```
START ─▶ agent ──(tool_calls 있음)──▶ tools ──▶ agent (다시 LLM)
              └─(없음, 최종 답변)────▶ END
```

이 루프를 LangGraph로 직접 짜면 **어디서 뭐가 돌아가는지 완벽하게 보인다**. (`create_react_agent`가 내부적으로 만들어주는 게 바로 이 그래프다.)

### 세 가지 조각

1. **`@tool` 데코레이터**로 도구 함수 정의 — docstring이 LLM에게 "이 도구를 언제 쓰는지" 알려준다.
2. **`llm.bind_tools([...])`** — LLM이 응답할 때 도구 호출(tool_calls)을 만들어낼 수 있게 한다.
3. **`ToolNode([...])`** — LLM이 요청한 tool_calls를 실제로 실행하는 노드 (LangGraph가 제공).
4. **`tools_condition`** — "tool_calls가 있으면 tools 노드로, 없으면 END로" 를 판단하는 **이미 만들어진 라우터 함수**.

### 이 노트북에서 쓸 도구
폐쇄망 친화적으로 외부 API 없이 로컬 도구 2개만 정의한다.
- `search_regulation(query)`: `corpus_ko.txt` 에서 간단 유사도 검색으로 1개 청크 반환.
- `today_iso()`: 오늘 날짜 문자열 반환.


In [ ]:
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from datetime import date

# --- 로컬 검색 인덱스 준비 (도구 내부에서 사용) ---
docs = load_any('../data/corpus_ko.txt')
chunks = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40).split_documents(docs)
vectordb = FAISS.from_documents(chunks, get_embeddings())

# --- 도구 정의 ---
@tool
def search_regulation(query: str) -> str:
    '''금융 규정 문서에서 주어진 query와 가장 관련된 조항/설명 한 조각을 찾아 반환한다.
    질문이 "전자금융거래법", "망분리", "개인정보" 처럼 국내 금융/규제 용어를 포함할 때 사용하라.
    '''
    hits = vectordb.similarity_search(query, k=1)
    return hits[0].page_content if hits else '(검색 결과 없음)'

@tool
def today_iso() -> str:
    '''오늘 날짜를 ISO 포맷(YYYY-MM-DD)으로 반환한다. 날짜·시점 관련 질문에 사용하라.'''
    return date.today().isoformat()

tools = [search_regulation, today_iso]
print('도구 등록:', [t.name for t in tools])


In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# --- LLM에 도구 바인딩 ---
llm_with_tools = llm.bind_tools(tools)

# --- 에이전트 노드: LLM이 답하거나 tool_calls를 만들어낸다 ---
def agent_node(state: AgentState) -> dict:
    reply = llm_with_tools.invoke(state['messages'])
    return {'messages': [reply]}

# --- 도구 실행 노드: LangGraph가 제공하는 prebuilt ---
tool_node = ToolNode(tools)

# --- 그래프 조립 ---
g = StateGraph(AgentState)
g.add_node('agent', agent_node)
g.add_node('tools', tool_node)

g.add_edge(START, 'agent')
g.add_conditional_edges(
    'agent',
    tools_condition,           # tool_calls 있으면 'tools', 없으면 END로 간다
    {'tools': 'tools', END: END},
)
g.add_edge('tools', 'agent')   # 도구 실행 결과를 다시 LLM에게 — 루프 닫기

agent_app = g.compile()
print('에이전트 그래프 컴파일 완료')


In [ ]:
# 실행: stream으로 각 단계를 관찰한다 — 도구 호출이 실제 일어나는지 눈으로 확인
question = '전자금융거래법이 규정하는 망분리 관련 내용을 알려주고, 오늘 날짜도 덧붙여줘.'

print('=== 에이전트 실행 trace ===')
for event in agent_app.stream({'messages': [HumanMessage(content=question)]}):
    for node_name, update in event.items():
        msg = update['messages'][-1]
        kind = type(msg).__name__
        # tool_calls가 있으면 내용 대신 호출 명세를 보여준다
        if getattr(msg, 'tool_calls', None):
            calls = [(c['name'], c['args']) for c in msg.tool_calls]
            print(f'[{node_name}] {kind} tool_calls={calls}')
        else:
            content = (msg.content or '')[:120]
            print(f'[{node_name}] {kind}: {content}')


> 💡 **`create_react_agent(llm, tools)` 한 줄이 내부적으로 만드는 그래프가 정확히 위와 같다.** 저수준으로 직접 짜보면 `tools_condition`이 뭘 판단하고, `ToolNode`가 어떤 순서로 실행되는지 전부 투명해진다. 이 패턴 자체는 본 과정에서 다시 쓰지 않지만, retrieval을 도구로 감싸 에이전트에게 맡기는 방식의 레퍼런스로 기억해두면 좋다.

## 8. 그래프 시각화: 내가 만든 흐름을 그림으로

복잡한 그래프를 만들수록 "머릿속 그래프 ≠ 실제 그래프" 문제가 자주 생긴다. 시각화는 1차 디버깅 도구다.

LangGraph의 컴파일된 앱은 `.get_graph()` 로 구조 정보를 꺼낼 수 있고, 거기서 mermaid 텍스트/ASCII 트리 등을 뽑아낼 수 있다.

- `draw_mermaid()` → mermaid.js 문자열. 주피터 마크다운 셀이나 mermaid.live에 붙여넣으면 그림으로 렌더된다.
- `draw_ascii()` → 텍스트 기반 다이어그램. 터미널에서도 확인 가능 (외부 패키지 불필요).


In [ ]:
# 1) Mermaid 포맷으로 뽑기 — 마크다운 뷰어에 복사해서 시각화 가능
mermaid_src = agent_app.get_graph().draw_mermaid()
print(mermaid_src)


In [ ]:
# 2) ASCII 포맷으로 터미널에서 바로 확인 (grandalf 설치 시에만 동작 — 실패해도 무방)
try:
    print(agent_app.get_graph().draw_ascii())
except Exception as e:
    print('ASCII 렌더 미지원:', e)
    print('→ 위의 mermaid_src를 https://mermaid.live 에 붙여넣어 시각화하세요.')


## 다음 단계: StateGraph로 실전 RAG 엮기

이 노트북에서 본 3요소(**State · Node · Edge**)와 **조건부 라우팅**이 앞으로 이어지는 RAG 노트북의 공통 뼈대가 된다.

| 노트북 | 이 노트북과의 연결 |
|---|---|
| `04_naive_rag_relevance.ipynb` | retrieve → 답변 생성 그래프에 **관련성 체크**를 조건부 엣지로 삽입 (그리고 무한 루프 문제를 일부러 드러낸다) |
| `05_self_rag_langgraph.ipynb` | grade → rewrite → retrieve 루프를 **조건부 엣지**로 구현해 04의 무한 루프를 깬다 |
| `06_adaptive_rag.ipynb` | 질문 복잡도로 **서로 다른 경로**로 라우팅한다 |

한 가지만 기억하자.

> **State는 계약, Node는 함수, Edge는 흐름.**

이 세 단어가 앞으로의 모든 그래프를 해석하는 렌즈다.